## Historical Profiling

Builds N profiles from real HUMMUS users: aggregates each selected user's past
ratings into a profile with an evidence trail, and marks inferred attributes
with a confidence level instead of asserting them as fact.

Deliberately does NOT infer a health "goal" (weight-loss, etc.) from ratings
alone — that would be exactly the unsupported inference the project plan warns
against. Goal is left "unknown" for every historical profile. The dietary-
constraint guess further down is included as an illustration of an evidence-
backed inference with an honest confidence label, not something to trust
blindly — check a few by hand before treating it as ground truth.

In [1]:
import json
import pandas as pd
import random

In [8]:
N_PROFILES = 12       # how many historical profiles to build
MIN_REVIEWS = 15      # minimum reviews to be eligible — tune after seeing Day 1 output
POSITIVE_RATING = 4   # ratings >= this count as "liked"

In [9]:
clean_recipes = pd.read_pickle("processed/clean_recipes.pkl")
reviews = pd.read_csv("data/pp_reviews.csv")

C:\Users\HP\AppData\Local\Temp\ipykernel_11608\2976645407.py:2: DtypeWarning: Columns (4,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("data/pp_reviews.csv")


In [10]:
for name, df in [("recipes", clean_recipes), ("reviews", reviews)]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())


=== recipes ===
shape: (507335, 11)
columns: ['new_recipe_id', 'title', 'tags_parsed', 'ingredients_parsed', 'duration', 'calories [cal]', 'sugars [g]', 'protein [g]', 'who_score', 'fsa_score', 'nutri_score']

=== reviews ===
shape: (1916424, 12)
columns: ['Unnamed: 0', 'member_id', 'recipe_id', 'rating', 'text', 'likes', 'last_modified_date', 'review_id', 'review_url', 'member_url', 'new_member_id', 'new_recipe_id']


In [24]:
# From the 'reviews' DataFrame
review_recipe_id_col = "new_recipe_id"
member_id_col        = "member_id"
rating_col           = "rating"

# From the 'clean_recipes' DataFrame
recipe_id_col        = "new_recipe_id"
tags_col             = "tags_parsed"
who_col              = "who_score"
fsa_col              = "fsa_score"
calories_col         = "calories [cal]"
sugars_col           = "sugars [g]"

In [25]:
review_counts = reviews.groupby(member_id_col).size()
eligible = review_counts[review_counts >= MIN_REVIEWS].index
print(f"{len(eligible)} users have >= {MIN_REVIEWS} reviews.")

13471 users have >= 15 reviews.


In [26]:
# Sample spread across the eligible range (not just the heaviest reviewers) for diversity
eligible_sorted = review_counts.loc[eligible].sort_values()
step = max(1, len(eligible_sorted) // N_PROFILES)
picked_members = eligible_sorted.iloc[::step].index[:N_PROFILES].tolist()
print("Picked member ids:", picked_members)

Picked member ids: [323832, 550647, 511719, 1906, 675125, 116478, 352042, 295324, 168753, 300888, 38865, 30503]


In [27]:
MEAT_TAGS = {"beef", "pork", "chicken", "poultry", "meat", "seafood", "fish", "turkey", "bacon", "ham"}

In [ ]:
def flatten_tags(tag_lists):
    out = []
    for t in tag_lists:
        if isinstance(t, list):
            out.extend(str(x).lower() for x in t)
    return out


profiles = []
for i, mid in enumerate(picked_members):
    user_reviews = reviews[reviews[member_id_col] == mid]
    merged = user_reviews.merge(
        clean_recipes, 
        left_on=recipe_id_col, 
        right_on=recipe_id_col, 
        how="inner"
        )
    liked = merged[merged[rating_col] >= POSITIVE_RATING]

    n_total, n_liked = len(merged), len(liked)
    evidence = merged[[recipe_id_col, rating_col]].rename(
        columns={recipe_id_col: "recipe_id", rating_col: "rating"}
    ).to_dict("records")

    attrs, confidence = {}, {}

    if tags_col and n_liked >= 5:
        tags_flat = flatten_tags(liked[tags_col])
        if tags_flat:
            top_tags = pd.Series(tags_flat).value_counts()
            attrs["top_tags"] = top_tags.head(5).index.tolist()
            confidence["top_tags"] = "high" if top_tags.iloc[0] / n_liked > 0.5 else "medium"

    for label, col in [("avg_calories", calories_col), ("avg_sugars", sugars_col)]:
        if col and n_liked >= 5:
            attrs[label] = round(float(liked[col].mean()), 1)
            confidence[label] = "medium"

    for label, col in [("avg_who_score", who_col), ("avg_fsa_score", fsa_col)]:
        if col and n_liked >= 5:
            attrs[label] = round(float(liked[col].mean()), 2)
            confidence[label] = "medium"

    if tags_col and n_liked >= 5:
        tags_flat = set(flatten_tags(liked[tags_col]))
        if tags_flat and not (tags_flat & MEAT_TAGS):
            attrs["possible_constraint"] = "vegetarian (inferred, unconfirmed)"
            confidence["possible_constraint"] = "low"

    attrs["goal"] = "unknown"
    confidence["goal"] = "no_evidence"

    profiles.append({
        "profile_id": f"hist_{i:03d}",
        "source": "historical",
        "member_id": str(mid),
        "n_interactions_total": n_total,
        "n_interactions_liked": n_liked,
        "attributes": attrs,
        "confidence": confidence,
        "evidence": evidence,
    })

In [29]:
with open("processed/historical_profiles.json", "w") as f:
    json.dump(profiles, f, indent=2, default=str)

print(f"\nSaved {len(profiles)} historical profiles -> {'processed/historical_profiles.json'}")
print("\nExample profile:")
print(json.dumps(profiles[0], indent=2, default=str))


Saved 12 historical profiles -> processed/historical_profiles.json

Example profile:
{
  "profile_id": "hist_000",
  "source": "historical",
  "member_id": "323832",
  "n_interactions_total": 15,
  "n_interactions_liked": 13,
  "attributes": {
    "top_tags": [
      "time-to-make",
      "course",
      "preparation",
      "dietary",
      "main-ingredient"
    ],
    "avg_calories": 273.2,
    "avg_sugars": 22.2,
    "avg_who_score": 0.22,
    "avg_fsa_score": 0.61,
    "goal": "unknown"
  },
  "confidence": {
    "top_tags": "high",
    "avg_calories": "medium",
    "avg_sugars": "medium",
    "avg_who_score": "medium",
    "avg_fsa_score": "medium",
    "goal": "no_evidence"
  },
  "evidence": [
    {
      "recipe_id": 299090,
      "rating": 5.0
    },
    {
      "recipe_id": 315294,
      "rating": 3.0
    },
    {
      "recipe_id": 340720,
      "rating": 5.0
    },
    {
      "recipe_id": 359568,
      "rating": 5.0
    },
    {
      "recipe_id": 415441,
      "rating": 

## Synthetic Profiling

Builds N synthetic profiles on the SAME schema as the historical ones, but with
attributes explicitly assigned rather than inferred. Includes a mix of clean
combinatorial profiles, two with deliberately conflicting priorities, and one
with almost no stated info — these edge cases are the ones most likely to
stress-test the LLM judge later.

In [2]:
N_PROFILES = 12
SEED = 7
random.seed(SEED)

TASTES = ["italian", "asian-inspired", "mexican", "mediterranean", "comfort-food", "baking-and-dessert", "spicy"]
GOALS = ["weight-loss", "muscle-gain", "heart-health", "general-wellness", "none-stated"]
TIME_BUDGETS = ["quick (<20 min)", "moderate (20-45 min)", "no-constraint"]
CONSTRAINTS = ["vegetarian", "vegan", "gluten-free", "nut-allergy", "low-sodium", "dairy-free", "none"]

In [3]:
profiles = []

# --- combinatorial "clean" profiles -------------------------------------------
n_clean = N_PROFILES - 3
for i in range(n_clean):
    attrs = {
        "top_tags": random.sample(TASTES, k=random.choice([1, 2])),
        "goal": random.choice(GOALS),
        "time_budget": random.choice(TIME_BUDGETS),
        "constraints": random.sample(CONSTRAINTS, k=random.choice([0, 1])) or ["none"],
    }
    profiles.append({
        "profile_id": f"synth_{i:03d}",
        "source": "synthetic",
        "attributes": attrs,
        "confidence": {k: "stated" for k in attrs},
        "evidence": [],
        "notes": "",
    })

In [4]:
# --- deliberately conflicting profiles -----------------------------------------
profiles.append({
    "profile_id": f"synth_{n_clean:03d}",
    "source": "synthetic",
    "attributes": {
        "top_tags": ["comfort-food"],
        "goal": "weight-loss",
        "time_budget": "no-constraint",
        "constraints": ["none"],
    },
    "confidence": {"top_tags": "stated", "goal": "stated", "time_budget": "stated", "constraints": "stated"},
    "evidence": [],
    "notes": "conflict: comfort-food taste vs. a stated weight-loss goal",
})

profiles.append({
    "profile_id": f"synth_{n_clean + 1:03d}",
    "source": "synthetic",
    "attributes": {
        "top_tags": ["baking-and-dessert"],
        "goal": "none-stated",
        "time_budget": "quick (<20 min)",
        "constraints": ["vegan"],
    },
    "confidence": {"top_tags": "stated", "goal": "stated", "time_budget": "stated", "constraints": "stated"},
    "evidence": [],
    "notes": "tension: dessert-focused taste under a vegan constraint",
})

In [6]:
# --- deliberately sparse / missing-info profile --------------------------------
profiles.append({
    "profile_id": f"synth_{n_clean + 2:03d}",
    "source": "synthetic",
    "attributes": {
        "top_tags": [],
        "goal": "none-stated",
        "time_budget": "no-constraint",
        "constraints": ["none"],
    },
    "confidence": {"top_tags": "no_evidence", "goal": "no_evidence", "time_budget": "stated", "constraints": "stated"},
    "evidence": [],
    "notes": "sparse profile: almost nothing stated, tests how judges handle ambiguity",
})

with open("processed/synthetic_profiles.json", "w") as f:
    json.dump(profiles, f, indent=2)

print(f"Saved {len(profiles)} synthetic profiles -> {'processed/synthetic_profiles.json'}")
print("\nExample clean profile:")
print(json.dumps(profiles[0], indent=2))
print("\nExample edge-case profile:")
print(json.dumps(profiles[-1], indent=2))

Saved 12 synthetic profiles -> processed/synthetic_profiles.json

Example clean profile:
{
  "profile_id": "synth_000",
  "source": "synthetic",
  "attributes": {
    "top_tags": [
      "asian-inspired",
      "mediterranean"
    ],
    "goal": "weight-loss",
    "time_budget": "quick (<20 min)",
    "constraints": [
      "none"
    ]
  },
  "confidence": {
    "top_tags": "stated",
    "goal": "stated",
    "time_budget": "stated",
    "constraints": "stated"
  },
  "evidence": [],
  "notes": ""
}

Example edge-case profile:
{
  "profile_id": "synth_011",
  "source": "synthetic",
  "attributes": {
    "top_tags": [],
    "goal": "none-stated",
    "time_budget": "no-constraint",
    "constraints": [
      "none"
    ]
  },
  "confidence": {
    "top_tags": "no_evidence",
    "goal": "no_evidence",
    "time_budget": "stated",
    "constraints": "stated"
  },
  "evidence": [],
  "notes": "sparse profile: almost nothing stated, tests how judges handle ambiguity"
}
